In [0]:
%run ../config/config-env

In [0]:
bronze_table =f'{catalog_name}.{bronze_schema}.sprints'
silver_table =f'{catalog_name}.{silver_schema}.sprints'

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_df = spark.read.table(bronze_table)

In [0]:
display(sprints_df)

In [0]:
sprints_dropped_df = sprints_df.drop('url')




display(sprints_dropped_df)

In [0]:
sprints_final_df = (
    sprints_dropped_df
        .withColumnsRenamed({
            'driverId': 'driver_id', 
            'raceName': 'race_name', 'positionText': 'finished_position_text',
            'date': 'race_date', 'laps': 'completed_laps', 'number': 'car_number',
            'position': 'finished_position'
        })
    
).filter(
    (F.col('season').isNotNull()) &
    (F.col('round').isNotNull() )&
    (F.col('driver_id').isNotNull())

).dropDuplicates().withColumn('race_name', F.initcap(F.col('race_name')))

display(sprints_renamed_df)

In [0]:
sprints_dropped_df.printSchema()

In [0]:
(
    sprints_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
display(sprints_final_df.count() - sprints_df.count())